# Kappa Foundation – Investment Policy Analysis

**Decision date:** 31 December 2022  
**Initial capital:** USD 12 million  
**Mandatory payments:** USD 2.4 million on 31 December 2024 and USD 3.6 million on 31 December 2027  
**2030 objective:** after the mandatory payments, preserve at least the purchasing power of USD 8 million in December-2022 dollars.

This notebook covers Sections 01–04 of the analysis: mandate and assumptions, bond/liability funding, market-data construction, and the initial portfolio decision.

**Information discipline.** Portfolio estimation uses data no later than 31 December 2022. January 2023–December 2025 is kept out of the decision process for later frozen-strategy evaluation.

**Decision discipline.** Model outputs are evidence, not automatic investment decisions. The growth allocation selected in Section 04 is therefore labelled **provisional** until the factor-model and out-of-sample evidence are completed.


In [ ]:
# ============================================================
# 01A. EXECUTIVE SETUP
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from scipy.optimize import brentq, minimize
from scipy.stats import norm
from pathlib import Path

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Project paths
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

for directory in [RAW_DIR, PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Case facts
DECISION_DATE = pd.Timestamp("2022-12-31")
INITIAL_CAPITAL = 12_000_000
LIABILITIES = {2024: 2_400_000, 2027: 3_600_000}
REAL_TARGET_2030 = 8_000_000
FACE_VALUE = 1_000

BOND_DATA = pd.DataFrame({
    "Bond": ["Alpha", "Beta", "Gamma", "Delta", "Epsilon"],
    "Maturity": [1, 2, 3, 5, 8],
    "CouponRate": [0.000, 0.000, 0.040, 0.035, 0.045],
    "Price": [958.77, 922.78, 1002.49, 986.07, 1040.71]
})

# Data windows required by the case
ESTIMATION_START = "2015-01-01"
ESTIMATION_END = "2022-12-31"
EVALUATION_START = "2023-01-01"
EVALUATION_END = "2025-12-31"

# Modelling assumptions — not case facts
BASE_INFLATION = 0.025
INFLATION_SENSITIVITY = [0.020, 0.025, 0.030]
BASE_RISK_AVERSION = 4
RISK_AVERSION_VALUES = [2, 4, 6, 8]

# Additional portfolio limits chosen to control concentration
MAX_STOCK_WEIGHT = 0.15
MAX_ETF_WEIGHT = 0.35

# Implementation-cost assumption; tested later
BASE_TRADING_COST = 0.001  # 10 bps one-way

display(BOND_DATA)


In [ ]:
# ============================================================
# 01B. CASE FACTS, ASSUMPTIONS AND 2030 TARGET
# ============================================================

Table_A1_Facts_vs_Assumptions = pd.DataFrame({
    "Item": [
        "Initial capital",
        "2024 payment",
        "2027 payment",
        "2030 real-value objective",
        "Trustee loss concern",
        "Base inflation",
        "Base risk aversion A",
        "Stock weight cap",
        "ETF weight cap",
        "Base one-way trading cost"
    ],
    "Value": [
        INITIAL_CAPITAL,
        LIABILITIES[2024],
        LIABILITIES[2027],
        REAL_TARGET_2030,
        "Losses approaching 15%; metric/horizon not specified",
        BASE_INFLATION,
        BASE_RISK_AVERSION,
        MAX_STOCK_WEIGHT,
        MAX_ETF_WEIGHT,
        BASE_TRADING_COST
    ],
    "Status": [
        "Case fact", "Case fact", "Case fact", "Case fact", "Case fact",
        "Model assumption", "Model assumption", "Additional limit",
        "Additional limit", "Implementation assumption"
    ]
})

inflation_rows = []
for inflation in INFLATION_SENSITIVITY:
    nominal_target = REAL_TARGET_2030 * (1 + inflation) ** 8
    inflation_rows.append({
        "InflationAssumption": inflation,
        "Nominal2030Target": nominal_target
    })

Table_A2_Inflation_Sensitivity = pd.DataFrame(inflation_rows)

display(Table_A1_Facts_vs_Assumptions)
display(Table_A2_Inflation_Sensitivity)


# 02. Bond Valuation & Liability Funding

The bond analysis addresses the required funding evidence directly:

- value the five case bonds and calculate YTM;
- infer the incomplete term structure, while making interpolation assumptions visible;
- calculate duration and convexity and check them against exact repricing;
- build a funded-payment plan for 2024 and 2027;
- test reinvestment assumptions for the 2027 liability;
- apply both a parallel rate shock and a curve-shape shock;
- distinguish mark-to-market price risk from contractual held-to-maturity funding ability.

The rate shocks are **stress scenarios**, not forecasts. A symmetric ±100 bp parallel shock is used as a material round-number rate move, while a ±75 bp slope shock tests curve-shape dependence without changing the average level of the curve.


In [ ]:
# ============================================================
# 02A. BOND FUNCTIONS AND TERM-STRUCTURE CONSTRUCTION
# ============================================================

def bond_cashflows(face, coupon_rate, maturity):
    """Return annual contractual bond cash flows."""
    coupon = face * coupon_rate

    cashflows = np.full(
        int(maturity),
        coupon,
        dtype=float
    )

    cashflows[-1] += face

    return cashflows


def price_from_ytm(
    face,
    coupon_rate,
    maturity,
    ytm
):
    """Price a bond using one annual effective YTM."""

    cashflows = bond_cashflows(
        face,
        coupon_rate,
        maturity
    )

    t = np.arange(
        1,
        int(maturity) + 1
    )

    return np.sum(
        cashflows
        / (1 + ytm) ** t
    )


def solve_ytm(
    face,
    coupon_rate,
    maturity,
    market_price
):
    """Solve the YTM that reproduces the quoted market price."""

    objective = lambda y: (
        price_from_ytm(
            face,
            coupon_rate,
            maturity,
            y
        )
        - market_price
    )

    return brentq(
        objective,
        -0.50,
        1.00
    )


def duration_convexity(
    face,
    coupon_rate,
    maturity,
    ytm
):
    """
    Calculate Macaulay duration, modified duration
    and standard discrete convexity.
    """

    cashflows = bond_cashflows(
        face,
        coupon_rate,
        maturity
    )

    t = np.arange(
        1,
        int(maturity) + 1
    )

    pv = (
        cashflows
        / (1 + ytm) ** t
    )

    price = pv.sum()

    macaulay = (
        np.sum(t * pv)
        / price
    )

    modified = (
        macaulay
        / (1 + ytm)
    )

    convexity = (
        np.sum(
            cashflows
            * t
            * (t + 1)
            / (1 + ytm) ** (t + 2)
        )
        / price
    )

    return (
        macaulay,
        modified,
        convexity
    )


def price_from_spot_curve(
    face,
    coupon_rate,
    maturity,
    spot_curve
):
    """Price contractual cash flows using maturity-specific spot rates."""

    cashflows = bond_cashflows(
        face,
        coupon_rate,
        maturity
    )

    price = 0.0

    for t in range(
        1,
        int(maturity) + 1
    ):
        price += (
            cashflows[t - 1]
            / (1 + spot_curve[t]) ** t
        )

    return price


# ------------------------------------------------------------
# Helper function: construct a complete 1Y-8Y spot curve
# ------------------------------------------------------------

def build_spot_curve(
    interpolation="spot"
):
    """
    Construct annual 1Y-8Y spot rates.

    interpolation = "spot":
        missing maturities are linearly interpolated in spot rates.

    interpolation = "discount":
        missing maturities are linearly interpolated in discount factors.
    """

    curve = {}

    # --------------------------------------------------------
    # 1Y: Alpha zero-coupon
    # --------------------------------------------------------

    alpha_price = BOND_DATA.loc[
        BOND_DATA["Bond"] == "Alpha",
        "Price"
    ].iloc[0]

    curve[1] = (
        FACE_VALUE / alpha_price
        - 1
    )

    # --------------------------------------------------------
    # 2Y: Beta zero-coupon
    # --------------------------------------------------------

    beta_price = BOND_DATA.loc[
        BOND_DATA["Bond"] == "Beta",
        "Price"
    ].iloc[0]

    curve[2] = (
        (FACE_VALUE / beta_price) ** (1 / 2)
        - 1
    )

    # --------------------------------------------------------
    # 3Y: Gamma bootstrap
    # --------------------------------------------------------

    gamma = BOND_DATA.loc[
        BOND_DATA["Bond"] == "Gamma"
    ].iloc[0]

    gamma_coupon = (
        FACE_VALUE
        * gamma["CouponRate"]
    )

    remaining_gamma_cf = (
        gamma["Price"]
        - gamma_coupon
          / (1 + curve[1])
        - gamma_coupon
          / (1 + curve[2]) ** 2
    )

    curve[3] = (
        (
            (FACE_VALUE + gamma_coupon)
            / remaining_gamma_cf
        ) ** (1 / 3)
        - 1
    )

    # --------------------------------------------------------
    # Helper for interpolation
    # --------------------------------------------------------

    def interpolate_between(
        t,
        t_left,
        r_left,
        t_right,
        r_right
    ):

        weight = (
            (t - t_left)
            / (t_right - t_left)
        )

        if interpolation == "spot":

            return (
                r_left
                + weight
                * (r_right - r_left)
            )

        elif interpolation == "discount":

            d_left = (
                1
                / (1 + r_left) ** t_left
            )

            d_right = (
                1
                / (1 + r_right) ** t_right
            )

            d_t = (
                d_left
                + weight
                * (d_right - d_left)
            )

            return (
                d_t ** (-1 / t)
                - 1
            )

        else:

            raise ValueError(
                "interpolation must be 'spot' or 'discount'"
            )

    # --------------------------------------------------------
    # Solve 5Y from Delta, with 4Y interpolated
    # --------------------------------------------------------

    delta = BOND_DATA.loc[
        BOND_DATA["Bond"] == "Delta"
    ].iloc[0]

    def delta_error(s5):

        s4 = interpolate_between(
            4,
            3,
            curve[3],
            5,
            s5
        )

        temp_curve = {
            1: curve[1],
            2: curve[2],
            3: curve[3],
            4: s4,
            5: s5
        }

        model_price = price_from_spot_curve(
            FACE_VALUE,
            delta["CouponRate"],
            int(delta["Maturity"]),
            temp_curve
        )

        return (
            model_price
            - delta["Price"]
        )

    curve[5] = brentq(
        delta_error,
        -0.25,
        0.50
    )

    curve[4] = interpolate_between(
        4,
        3,
        curve[3],
        5,
        curve[5]
    )

    # --------------------------------------------------------
    # Solve 8Y from Epsilon, with 6Y and 7Y interpolated
    # --------------------------------------------------------

    epsilon = BOND_DATA.loc[
        BOND_DATA["Bond"] == "Epsilon"
    ].iloc[0]

    def epsilon_error(s8):

        s6 = interpolate_between(
            6,
            5,
            curve[5],
            8,
            s8
        )

        s7 = interpolate_between(
            7,
            5,
            curve[5],
            8,
            s8
        )

        temp_curve = {
            1: curve[1],
            2: curve[2],
            3: curve[3],
            4: curve[4],
            5: curve[5],
            6: s6,
            7: s7,
            8: s8
        }

        model_price = price_from_spot_curve(
            FACE_VALUE,
            epsilon["CouponRate"],
            int(epsilon["Maturity"]),
            temp_curve
        )

        return (
            model_price
            - epsilon["Price"]
        )

    curve[8] = brentq(
        epsilon_error,
        -0.25,
        0.50
    )

    curve[6] = interpolate_between(
        6,
        5,
        curve[5],
        8,
        curve[8]
    )

    curve[7] = interpolate_between(
        7,
        5,
        curve[5],
        8,
        curve[8]
    )

    return curve


# Base term structure
spot_curve = build_spot_curve(
    interpolation="spot"
)

spot_curve_discount_interp = build_spot_curve(
    interpolation="discount"
)


In [ ]:
# ============================================================
# 02B. BOND VALUATION, YIELD ANALYSIS AND EXACT REPRICING CHECK
# ============================================================

# Local +/-100 bp check used only to validate duration-convexity approximation.
DC_CHECK_SHOCK = 0.01


bond_results = []
dc_check_results = []

for _, row in BOND_DATA.iterrows():

    bond_name = row["Bond"]
    maturity = int(row["Maturity"])
    coupon_rate = row["CouponRate"]
    market_price = row["Price"]

    # --------------------------------------------------------
    # YTM
    # --------------------------------------------------------

    ytm = solve_ytm(
        FACE_VALUE,
        coupon_rate,
        maturity,
        market_price
    )

    # --------------------------------------------------------
    # Duration and convexity
    # --------------------------------------------------------

    (
        macaulay,
        modified,
        convexity
    ) = duration_convexity(
        FACE_VALUE,
        coupon_rate,
        maturity,
        ytm
    )

    # --------------------------------------------------------
    # Spot-curve repricing check
    # --------------------------------------------------------

    spot_reprice = price_from_spot_curve(
        FACE_VALUE,
        coupon_rate,
        maturity,
        spot_curve
    )

    bond_results.append({

        "Bond": bond_name,
        "Maturity": maturity,
        "CouponRate": coupon_rate,
        "Price": market_price,
        "YTM": ytm,
        "MacaulayDuration": macaulay,
        "ModifiedDuration": modified,
        "Convexity": convexity,
        "SpotCurveReprice": spot_reprice,
        "SpotRepricingError": (
            spot_reprice
            - market_price
        )
    })

    # --------------------------------------------------------
    # Duration/convexity validation:
    # compare with exact YTM repricing under +/-100 bp
    # --------------------------------------------------------

    for dy in [
        -DC_CHECK_SHOCK,
        DC_CHECK_SHOCK
    ]:

        exact_price = price_from_ytm(
            FACE_VALUE,
            coupon_rate,
            maturity,
            ytm + dy
        )

        approx_pct_change = (
            -modified * dy
            + 0.5
            * convexity
            * dy ** 2
        )

        approx_price = (
            market_price
            * (1 + approx_pct_change)
        )

        dc_check_results.append({

            "Bond": bond_name,
            "YieldShock": dy,
            "BaseYTM": ytm,
            "ExactRepricedPrice": exact_price,
            "DurationConvexityPrice": approx_price,
            "ApproximationError": (
                approx_price
                - exact_price
            ),
            "ApproximationErrorPct": (
                approx_price
                / exact_price
                - 1
            )
        })


Table_B1_Bond_Analytics = pd.DataFrame(
    bond_results
)

Table_B1c_Duration_Convexity_Check = pd.DataFrame(
    dc_check_results
)


# ------------------------------------------------------------
# Term-structure table:
# clearly distinguish observed / bootstrapped / interpolated
# ------------------------------------------------------------

rate_source = {
    1: "Observed from Alpha zero",
    2: "Observed from Beta zero",
    3: "Bootstrapped from Gamma",
    4: "Interpolated",
    5: "Solved from Delta",
    6: "Interpolated",
    7: "Interpolated",
    8: "Solved from Epsilon"
}

Table_B1b_Term_Structure = pd.DataFrame({

    "Maturity": range(1, 9),

    "SpotRate": [
        spot_curve[t]
        for t in range(1, 9)
    ],

    "Source": [
        rate_source[t]
        for t in range(1, 9)
    ]
})


display(Table_B1_Bond_Analytics)
display(Table_B1b_Term_Structure)
display(Table_B1c_Duration_Convexity_Check)


In [ ]:
# ============================================================
# 02C. LIABILITY FUNDING AND FUNDING ADEQUACY
# ============================================================

# 2024 liability: exact nominal match with 2Y zero-coupon Beta
beta = BOND_DATA.loc[BOND_DATA["Bond"] == "Beta"].iloc[0]
beta_units = LIABILITIES[2024] / FACE_VALUE
beta_initial_cost = beta_units * beta["Price"]
beta_contractual_cash_2024 = beta_units * FACE_VALUE

# 2027 liability: Delta coupons ring-fenced as cash; 0% base reinvestment
delta = BOND_DATA.loc[BOND_DATA["Bond"] == "Delta"].iloc[0]
delta_cashflows = bond_cashflows(
    FACE_VALUE, delta["CouponRate"], int(delta["Maturity"])
)
delta_accumulated_cash_per_bond_2027 = delta_cashflows.sum()
delta_units = LIABILITIES[2027] / delta_accumulated_cash_per_bond_2027
delta_initial_cost = delta_units * delta["Price"]
delta_contractual_cash_2027 = (
    delta_units * delta_accumulated_cash_per_bond_2027
)

LIABILITY_BUCKET_COST = beta_initial_cost + delta_initial_cost
GROWTH_BUDGET_BEFORE_COSTS = INITIAL_CAPITAL - LIABILITY_BUCKET_COST

NOMINAL_TARGET_2030 = REAL_TARGET_2030 * (1 + BASE_INFLATION) ** 8
ROUGH_REQUIRED_GROWTH_CAGR = (
    NOMINAL_TARGET_2030 / GROWTH_BUDGET_BEFORE_COSTS
) ** (1 / 8) - 1

Table_B2_Funding_Plan = pd.DataFrame({
    "Liability": ["Scholarship programme", "Learning centre"],
    "PaymentYear": [2024, 2027],
    "RequiredPayment": [LIABILITIES[2024], LIABILITIES[2027]],
    "FundingAsset": ["Beta", "Delta"],
    "Units": [beta_units, delta_units],
    "InitialCost": [beta_initial_cost, delta_initial_cost],
    "FundingMethod": [
        "2Y zero-coupon maturity match",
        "Delta coupons ring-fenced; 0% base reinvestment"
    ]
})

Table_B2b_Funding_Adequacy = pd.DataFrame({
    "Liability": ["2024 Scholarship", "2027 Learning Centre"],
    "RequiredPayment": [LIABILITIES[2024], LIABILITIES[2027]],
    "ContractualFundedCash": [
        beta_contractual_cash_2024,
        delta_contractual_cash_2027
    ],
    "FundingRatio": [
        beta_contractual_cash_2024 / LIABILITIES[2024],
        delta_contractual_cash_2027 / LIABILITIES[2027]
    ],
    "PrimaryResidualRisk": [
        "Mark-to-market movement does not impair maturity cash flow if held",
        "Requires ring-fencing of interim coupons; reinvestment rate tested separately"
    ]
})

Table_B2d_Target_Diagnostic = pd.DataFrame({
    "Metric": [
        "Liability reserve cost",
        "Residual growth budget before trading costs",
        "2030 nominal target at base inflation",
        "Rough growth-sleeve CAGR diagnostic"
    ],
    "Value": [
        LIABILITY_BUCKET_COST,
        GROWTH_BUDGET_BEFORE_COSTS,
        NOMINAL_TARGET_2030,
        ROUGH_REQUIRED_GROWTH_CAGR
    ],
    "Interpretation": [
        "Up-front amount dedicated to mandatory payments",
        "Capital available for growth before implementation cost",
        "Real USD 8m target translated to nominal dollars",
        "Diagnostic only; not a promised total-fund return"
    ]
})

display(Table_B2_Funding_Plan)
display(Table_B2b_Funding_Adequacy)
display(Table_B2d_Target_Diagnostic)


In [ ]:
# ============================================================
# 02D. REINVESTMENT-RATE SENSITIVITY FOR 2027 LIABILITY
# ============================================================

def accumulated_delta_value(
    reinvestment_rate
):
    """
    Future value at end-2027 of one Delta bond,
    assuming each interim coupon is reinvested at
    one constant annual rate until 2027.
    """

    maturity = int(
        delta["Maturity"]
    )

    cashflows = bond_cashflows(
        FACE_VALUE,
        delta["CouponRate"],
        maturity
    )

    future_value = 0.0

    for t, cf in enumerate(
        cashflows,
        start=1
    ):

        years_to_2027 = (
            maturity - t
        )

        future_value += (
            cf
            * (1 + reinvestment_rate)
            ** years_to_2027
        )

    return future_value


reinvestment_scenarios = {
    "0% cash base case": 0.00,
    "2% reinvestment": 0.02,
    "4% reinvestment": 0.04
}


reinvestment_results = []

for scenario, rate in reinvestment_scenarios.items():

    fv_per_bond = (
        accumulated_delta_value(rate)
    )

    units_required = (
        LIABILITIES[2027]
        / fv_per_bond
    )

    initial_cost = (
        units_required
        * delta["Price"]
    )

    reinvestment_results.append({

        "Scenario": scenario,
        "ReinvestmentRate": rate,
        "FutureValuePerDelta": fv_per_bond,
        "DeltaUnitsRequired": units_required,
        "InitialFundingCost": initial_cost
    })


Table_B2c_Reinvestment_Sensitivity = pd.DataFrame(
    reinvestment_results
)

display(
    Table_B2c_Reinvestment_Sensitivity
)


In [ ]:
# ============================================================
# 02E. PARALLEL AND CURVE-SHAPE RATE SCENARIOS
# ============================================================

PARALLEL_SHOCK = 0.0100      # +/-100 bp stress
CURVE_SHAPE_SHOCK = 0.0075  # +/-75 bp slope stress

Table_B3a_Scenario_Rationale = pd.DataFrame({
    "ScenarioInput": ["Parallel shock", "Curve-shape shock"],
    "Magnitude": [PARALLEL_SHOCK, CURVE_SHAPE_SHOCK],
    "Rationale": [
        "Symmetric material stress to test duration exposure; not a forecast",
        "Changes short-vs-long rates while leaving the average curve level broadly unchanged"
    ]
})

def shift_spot_curve(base_curve, parallel_shift=0.0, slope_shift=0.0):
    maturities = sorted(base_curve.keys())
    midpoint = (min(maturities) + max(maturities)) / 2
    half_range = (max(maturities) - min(maturities)) / 2

    shocked_curve = {}
    for t in maturities:
        slope_loading = (t - midpoint) / half_range
        shocked_curve[t] = (
            base_curve[t]
            + parallel_shift
            + slope_shift * slope_loading
        )
    return shocked_curve

curve_scenarios = {
    "Base": {"parallel": 0.0000, "slope": 0.0000},
    "Parallel +100bp": {"parallel": PARALLEL_SHOCK, "slope": 0.0000},
    "Parallel -100bp": {"parallel": -PARALLEL_SHOCK, "slope": 0.0000},
    "Steepener": {"parallel": 0.0000, "slope": CURVE_SHAPE_SHOCK},
    "Flattener": {"parallel": 0.0000, "slope": -CURVE_SHAPE_SHOCK}
}

curve_scenario_results = []
funding_scenario_results = []

for scenario_name, shocks in curve_scenarios.items():
    shocked_curve = shift_spot_curve(
        spot_curve,
        parallel_shift=shocks["parallel"],
        slope_shift=shocks["slope"]
    )

    for _, row in BOND_DATA.iterrows():
        exact_price = price_from_spot_curve(
            FACE_VALUE,
            row["CouponRate"],
            int(row["Maturity"]),
            shocked_curve
        )
        curve_scenario_results.append({
            "Scenario": scenario_name,
            "Bond": row["Bond"],
            "BasePrice": row["Price"],
            "ExactRepricedPrice": exact_price,
            "PriceChangePct": exact_price / row["Price"] - 1
        })

    beta_price_s = price_from_spot_curve(
        FACE_VALUE, beta["CouponRate"], int(beta["Maturity"]), shocked_curve
    )
    delta_price_s = price_from_spot_curve(
        FACE_VALUE, delta["CouponRate"], int(delta["Maturity"]), shocked_curve
    )

    funding_scenario_results.append({
        "Scenario": scenario_name,
        "BetaMarketValue": beta_units * beta_price_s,
        "DeltaMarketValue": delta_units * delta_price_s,
        "TotalLiabilitySleeveMarketValue": (
            beta_units * beta_price_s + delta_units * delta_price_s
        ),
        "2024ContractualFundingRatioIfHeld": 1.0,
        "2027ContractualFundingRatioIfHeldAndCouponsRingFenced": 1.0
    })

Table_B3_Rate_Scenarios = pd.DataFrame(curve_scenario_results)
Table_B3b_Funding_Under_Rate_Scenarios = pd.DataFrame(funding_scenario_results)

display(Table_B3a_Scenario_Rationale)
display(Table_B3_Rate_Scenarios)
display(Table_B3b_Funding_Under_Rate_Scenarios)


In [ ]:
# ============================================================
# 02G. INCOMPLETE TERM-STRUCTURE ROBUSTNESS
# ============================================================

interpolation_results = []


for method, curve in {

    "Linear spot-rate interpolation":
        spot_curve,

    "Linear discount-factor interpolation":
        spot_curve_discount_interp

}.items():

    # Reprice all five case bonds
    max_abs_error = 0.0

    for _, row in BOND_DATA.iterrows():

        model_price = (
            price_from_spot_curve(
                FACE_VALUE,
                row["CouponRate"],
                int(row["Maturity"]),
                curve
            )
        )

        pricing_error = (
            model_price
            - row["Price"]
        )

        max_abs_error = max(
            max_abs_error,
            abs(pricing_error)
        )

    interpolation_results.append({

        "InterpolationMethod": method,

        "Spot4Y": curve[4],
        "Spot6Y": curve[6],
        "Spot7Y": curve[7],

        "MaxAbsoluteBondRepricingError":
            max_abs_error
    })


Table_B4_Interpolation_Robustness = pd.DataFrame(
    interpolation_results
)

display(
    Table_B4_Interpolation_Robustness
)


In [ ]:
# ============================================================
# 02H. SECTION 02 SUMMARY AND EXPORT
# ============================================================

Section_02_Summary = pd.DataFrame({
    "Item": [
        "2024 contractual funding ratio",
        "2027 contractual funding ratio",
        "Liability reserve cost",
        "Growth budget before trading costs",
        "Base 2030 nominal target",
        "Rough growth-sleeve CAGR diagnostic"
    ],
    "Value": [
        beta_contractual_cash_2024 / LIABILITIES[2024],
        delta_contractual_cash_2027 / LIABILITIES[2027],
        LIABILITY_BUCKET_COST,
        GROWTH_BUDGET_BEFORE_COSTS,
        NOMINAL_TARGET_2030,
        ROUGH_REQUIRED_GROWTH_CAGR
    ]
})

section_02_outputs = {
    "Table_B1_Bond_Analytics": Table_B1_Bond_Analytics,
    "Table_B1b_Term_Structure": Table_B1b_Term_Structure,
    "Table_B1c_Duration_Convexity_Check": Table_B1c_Duration_Convexity_Check,
    "Table_B2_Funding_Plan": Table_B2_Funding_Plan,
    "Table_B2b_Funding_Adequacy": Table_B2b_Funding_Adequacy,
    "Table_B2c_Reinvestment_Sensitivity": Table_B2c_Reinvestment_Sensitivity,
    "Table_B2d_Target_Diagnostic": Table_B2d_Target_Diagnostic,
    "Table_B3a_Scenario_Rationale": Table_B3a_Scenario_Rationale,
    "Table_B3_Rate_Scenarios": Table_B3_Rate_Scenarios,
    "Table_B3b_Funding_Under_Rate_Scenarios": Table_B3b_Funding_Under_Rate_Scenarios,
    "Table_B4_Interpolation_Robustness": Table_B4_Interpolation_Robustness,
    "Section_02_Summary": Section_02_Summary
}

for name, table in section_02_outputs.items():
    table.to_csv(TABLE_DIR / f"{name}.csv", index=False)

display(Section_02_Summary)


# 03. Market Data, Universe & Reproducibility

This section establishes the investable growth universe and the exact data used later.

The universe contains **10 U.S. common stocks across at least four sectors plus two U.S.-listed U.S.-equity ETFs with different mandates**. SPY is also used as the simple investable growth benchmark and market proxy.

The decision sample is January 2015–December 2022. January 2023–December 2025 is isolated for later evaluation and is not used to select or estimate the initial portfolio.

Historical price coverage is used to verify that each ticker existed before the case decision date. This does **not** eliminate retrospective-selection or survivorship bias; the universe is therefore disclosed as a deliberately selected case universe rather than claimed to be an unbiased sample of all U.S. securities.


In [ ]:
# ============================================================
# 03A. CANDIDATE UNIVERSE
# ============================================================

STOCKS = [
    "AAPL", "MSFT", "JPM", "JNJ", "PG",
    "HD", "CAT", "CVX", "KO", "WMT"
]

ETFS = ["SPY", "IWM"]
BENCHMARK = "SPY"
CANDIDATE_TICKERS = STOCKS + ETFS

sector_or_mandate = {
    "AAPL": "Information Technology",
    "MSFT": "Information Technology",
    "JPM": "Financials",
    "JNJ": "Health Care",
    "PG": "Consumer Staples",
    "HD": "Consumer Discretionary",
    "CAT": "Industrials",
    "CVX": "Energy",
    "KO": "Consumer Staples",
    "WMT": "Consumer Staples",
    "SPY": "Broad U.S. large-cap equity ETF",
    "IWM": "U.S. small-cap equity ETF"
}

security_type = {
    ticker: ("ETF" if ticker in ETFS else "Common stock")
    for ticker in CANDIDATE_TICKERS
}

Table_D1_Candidate_Universe = pd.DataFrame({
    "Ticker": CANDIDATE_TICKERS,
    "SecurityType": [security_type[t] for t in CANDIDATE_TICKERS],
    "SectorOrMandate": [sector_or_mandate[t] for t in CANDIDATE_TICKERS],
    "PortfolioRole": [
        "Growth/diversification" if t != "SPY"
        else "Broad-market core; also benchmark and market proxy"
        for t in CANDIDATE_TICKERS
    ]
})

display(Table_D1_Candidate_Universe)


In [ ]:
# ============================================================
# 03B. LOAD SAVED RAW MARKET DATA AND SOURCE RECORD
# ============================================================

RAW_MARKET_FILE = RAW_DIR / "yahoo_market_data_raw.pkl"
SOURCE_RECORD_FILE = RAW_DIR / "yahoo_source_record.csv"

if not RAW_MARKET_FILE.exists():
    raise FileNotFoundError(
        "Missing saved Yahoo raw file. Run src/01_download_market_data.py first."
    )

market_raw = pd.read_pickle(RAW_MARKET_FILE)
Table_D2_Data_Record = pd.read_csv(SOURCE_RECORD_FILE)

print("Raw market-data shape:", market_raw.shape)
print("Raw date range:", market_raw.index.min(), "to", market_raw.index.max())
display(Table_D2_Data_Record)


In [ ]:
# ============================================================
# 03C. RAW DATA QUALITY, CASE-DATE ELIGIBILITY AND COVERAGE
# ============================================================

if not isinstance(market_raw.columns, pd.MultiIndex):
    raise ValueError("Expected Yahoo raw data to have MultiIndex columns.")

available_fields = market_raw.columns.get_level_values(0).unique().tolist()
available_tickers = market_raw.columns.get_level_values(1).unique().tolist()

missing_tickers = [t for t in CANDIDATE_TICKERS if t not in available_tickers]
if missing_tickers:
    raise ValueError(f"Missing candidate tickers in Yahoo raw data: {missing_tickers}")

if "Adj Close" not in available_fields:
    raise ValueError("Adj Close is missing; return construction must not continue.")

adjusted_close_daily = market_raw["Adj Close"][CANDIDATE_TICKERS].copy()

coverage_rows = []
for ticker in CANDIDATE_TICKERS:
    first_valid = adjusted_close_daily[ticker].first_valid_index()
    last_valid = adjusted_close_daily[ticker].last_valid_index()

    coverage_rows.append({
        "Ticker": ticker,
        "FirstValidPriceDate": first_valid,
        "LastValidPriceDate": last_valid,
        "HistoricalPriceExistsBeforeDecisionDate": (
            first_valid is not None and first_valid <= DECISION_DATE
        ),
        "SecurityTypeOrMandate": sector_or_mandate[ticker],
        "SelectionBiasNote": (
            "Historical existence verified; retrospective universe selection still disclosed"
        )
    })

Table_D3_Eligibility_and_Coverage = pd.DataFrame(coverage_rows)

if not Table_D3_Eligibility_and_Coverage[
    "HistoricalPriceExistsBeforeDecisionDate"
].all():
    raise ValueError("At least one candidate lacks historical price evidence before the case date.")

display(Table_D3_Eligibility_and_Coverage)


In [ ]:
# ============================================================
# 03D. MONTHLY RETURN CONSTRUCTION
# ============================================================

monthly_prices = adjusted_close_daily.resample("ME").last()

# Simple monthly adjusted-price returns.
# Yahoo Adjusted Close is used with auto_adjust=False in the retrieval script.
monthly_returns = monthly_prices.pct_change(fill_method=None)

estimation_returns_precheck = monthly_returns.loc[
    "2015-01-31":"2022-12-31",
    CANDIDATE_TICKERS
].copy()

evaluation_returns_precheck = monthly_returns.loc[
    "2023-01-31":"2025-12-31",
    CANDIDATE_TICKERS
].copy()

Table_D4_Missing_Data = pd.DataFrame({
    "Ticker": CANDIDATE_TICKERS,
    "EstimationMissing": [
        estimation_returns_precheck[t].isna().sum() for t in CANDIDATE_TICKERS
    ],
    "EvaluationMissing": [
        evaluation_returns_precheck[t].isna().sum() for t in CANDIDATE_TICKERS
    ]
})

# Common complete monthly samples — no imputation or invented returns
estimation_returns = estimation_returns_precheck.dropna(how="any")
evaluation_returns = evaluation_returns_precheck.dropna(how="any")

if len(estimation_returns) < 60:
    raise ValueError("Fewer than 60 common monthly estimation observations.")

if estimation_returns.index.max() > DECISION_DATE:
    raise ValueError("Post-2022 information leaked into the estimation sample.")

print("Estimation observations:", len(estimation_returns))
print("Evaluation observations:", len(evaluation_returns))
display(Table_D4_Missing_Data)


In [ ]:
# ============================================================
# 03E. RETURN-CONSTRUCTION AND REPRODUCIBILITY RECORD
# ============================================================

Table_D5_Return_Construction = pd.DataFrame({
    "Item": [
        "Market-data source",
        "Raw frequency",
        "Monthly sampling",
        "Return definition",
        "Corporate-action treatment",
        "Estimation window",
        "Evaluation window",
        "Missing-data treatment",
        "Units",
        "Benchmark",
        "Retrospective-selection limitation"
    ],
    "Treatment": [
        "Yahoo Finance via yfinance; saved untransformed response",
        "Daily",
        "Last available adjusted close in each calendar month",
        "P_adj,t / P_adj,t-1 - 1",
        "Yahoo Adjusted Close; auto_adjust=False in retrieval script",
        "Jan 2015-Dec 2022",
        "Jan 2023-Dec 2025; isolated from decision estimation",
        "Common complete months only; no imputation",
        "Decimal monthly returns",
        "SPY",
        "Universe selected retrospectively for the case; historical existence does not remove selection/survivorship bias"
    ]
})

Table_D6_Return_Diagnostics = pd.DataFrame({
    "Ticker": CANDIDATE_TICKERS,
    "Obs": [estimation_returns[t].count() for t in CANDIDATE_TICKERS],
    "MeanMonthly": [estimation_returns[t].mean() for t in CANDIDATE_TICKERS],
    "StdMonthly": [estimation_returns[t].std(ddof=1) for t in CANDIDATE_TICKERS],
    "MinMonthly": [estimation_returns[t].min() for t in CANDIDATE_TICKERS],
    "MaxMonthly": [estimation_returns[t].max() for t in CANDIDATE_TICKERS]
})

display(Table_D5_Return_Construction)
display(Table_D6_Return_Diagnostics)


In [ ]:
# ============================================================
# 03F. SAVE PROCESSED DATA AND SECTION OUTPUTS
# ============================================================

monthly_prices.to_csv(PROCESSED_DIR / "monthly_adjusted_prices.csv")
monthly_returns.to_csv(PROCESSED_DIR / "monthly_returns_full.csv")
estimation_returns.to_csv(
    PROCESSED_DIR / "monthly_returns_estimation_2015_2022.csv"
)
evaluation_returns.to_csv(
    PROCESSED_DIR / "monthly_returns_evaluation_2023_2025.csv"
)

section_03_outputs = {
    "Table_D1_Candidate_Universe": Table_D1_Candidate_Universe,
    "Table_D2_Data_Record": Table_D2_Data_Record,
    "Table_D3_Eligibility_and_Coverage": Table_D3_Eligibility_and_Coverage,
    "Table_D4_Missing_Data": Table_D4_Missing_Data,
    "Table_D5_Return_Construction": Table_D5_Return_Construction,
    "Table_D6_Return_Diagnostics": Table_D6_Return_Diagnostics
}

for name, table in section_03_outputs.items():
    table.to_csv(TABLE_DIR / f"{name}.csv", index=False)

print("Section 03 processed data and named outputs saved.")


# 04. Portfolio Construction & Risk Preferences

This section determines the **provisional growth allocation** after funding the mandatory payments.

To make the required model comparison meaningful, the candidate universe, expected-return inputs, risk-aversion parameter and constraints are held fixed while only the covariance estimator changes:

1. sample covariance; and
2. single-index covariance using SPY as the market proxy.

Expected returns use a CAPM-style common systematic framework estimated only from information through December 2022. Historical asset means are retained as an explicit sensitivity case because expected-return estimation is uncertain.

The trustees' concern about losses approaching 15% is treated as a **decision diagnostic**, not a guaranteed loss limit, because the case does not specify a metric or horizon.


In [ ]:
# ============================================================
# 04A. ALIGN YAHOO RETURNS WITH PRE-2023 FRENCH FACTORS
# ============================================================

FRENCH_ESTIMATION_FILE = (
    PROCESSED_DIR / "french_factors_estimation_2015_2022.csv"
)

if not FRENCH_ESTIMATION_FILE.exists():
    raise FileNotFoundError(
        "Missing French estimation file. Run src/02_download_french_factors.py first."
    )

french_estimation = pd.read_csv(
    FRENCH_ESTIMATION_FILE,
    index_col="Date",
    parse_dates=True
)

common_factor_dates = estimation_returns.index.intersection(
    french_estimation.index
)

returns_for_model = estimation_returns.loc[
    common_factor_dates, CANDIDATE_TICKERS
].copy()

factors_for_model = french_estimation.loc[common_factor_dates].copy()

if len(common_factor_dates) < 60:
    raise ValueError("Fewer than 60 common Yahoo/French monthly observations.")

if common_factor_dates.max() > DECISION_DATE:
    raise ValueError("Post-2022 factor information leaked into estimation.")

print("Common model observations:", len(common_factor_dates))
print("Model range:", common_factor_dates.min(), "to", common_factor_dates.max())


In [ ]:
# ============================================================
# 04B. EXPECTED-RETURN INPUTS
# ============================================================

rf_monthly = factors_for_model["RF"]
market_excess_monthly = factors_for_model["Mkt-RF"]

BASE_RF_ANNUAL = rf_monthly.mean() * 12
BASE_MARKET_PREMIUM_ANNUAL = market_excess_monthly.mean() * 12

expected_return_records = []

for ticker in CANDIDATE_TICKERS:
    asset_return = returns_for_model[ticker]
    asset_excess = asset_return - rf_monthly

    X = sm.add_constant(market_excess_monthly)
    capm_model = sm.OLS(asset_excess, X).fit()

    beta = capm_model.params["Mkt-RF"]
    capm_style_return = (
        BASE_RF_ANNUAL + beta * BASE_MARKET_PREMIUM_ANNUAL
    )

    expected_return_records.append({
        "Ticker": ticker,
        "CAPMBeta": beta,
        "CAPMAlphaMonthly": capm_model.params["const"],
        "CAPMStyleExpectedReturn": capm_style_return,
        "HistoricalAnnualMean": asset_return.mean() * 12
    })

Table_P1_Expected_Return_Inputs = (
    pd.DataFrame(expected_return_records).set_index("Ticker")
)

BASE_EXPECTED_RETURNS = Table_P1_Expected_Return_Inputs[
    "CAPMStyleExpectedReturn"
]
ALTERNATIVE_EXPECTED_RETURNS = Table_P1_Expected_Return_Inputs[
    "HistoricalAnnualMean"
]

Table_P1b_Return_Assumptions = pd.DataFrame({
    "Input": [
        "Annualized risk-free input",
        "Annualized market excess-return input",
        "Base expected-return model",
        "Alternative expected-return model"
    ],
    "Value": [
        BASE_RF_ANNUAL,
        BASE_MARKET_PREMIUM_ANNUAL,
        "CAPM-style: RF + beta x market premium",
        "Asset historical arithmetic mean, sensitivity only"
    ],
    "Caveat": [
        "Estimated from 2015-2022 French RF",
        "Estimated from 2015-2022 French Mkt-RF; not a promised future premium",
        "Common systematic framework; still dependent on historical inputs",
        "Noisy and highly optimizer-sensitive"
    ]
})

display(Table_P1_Expected_Return_Inputs)
display(Table_P1b_Return_Assumptions)


In [ ]:
# ============================================================
# 04C. SAMPLE AND SINGLE-INDEX RISK MODELS
# ============================================================

candidate_returns = returns_for_model[CANDIDATE_TICKERS].copy()

SAMPLE_COV = candidate_returns.cov() * 12

market_returns = returns_for_model[BENCHMARK]
market_variance_monthly = market_returns.var(ddof=1)

si_betas = {}
si_residual_variances = {}

for ticker in CANDIDATE_TICKERS:
    if ticker == BENCHMARK:
        si_betas[ticker] = 1.0
        si_residual_variances[ticker] = 0.0
        continue

    X = sm.add_constant(market_returns)
    model = sm.OLS(candidate_returns[ticker], X).fit()

    si_betas[ticker] = model.params[BENCHMARK]
    si_residual_variances[ticker] = model.resid.var(ddof=1)

SINGLE_INDEX_COV_MONTHLY = pd.DataFrame(
    index=CANDIDATE_TICKERS,
    columns=CANDIDATE_TICKERS,
    dtype=float
)

for i in CANDIDATE_TICKERS:
    for j in CANDIDATE_TICKERS:
        systematic = (
            si_betas[i] * si_betas[j] * market_variance_monthly
        )
        SINGLE_INDEX_COV_MONTHLY.loc[i, j] = (
            systematic
            + (si_residual_variances[i] if i == j else 0.0)
        )

SINGLE_INDEX_COV = SINGLE_INDEX_COV_MONTHLY * 12

Table_P2_Risk_Model_Diagnostics = pd.DataFrame({
    "RiskModel": ["Sample covariance", "Single-index covariance"],
    "MinimumEigenvalue": [
        np.linalg.eigvalsh(SAMPLE_COV.values).min(),
        np.linalg.eigvalsh(SINGLE_INDEX_COV.values).min()
    ],
    "MaximumEigenvalue": [
        np.linalg.eigvalsh(SAMPLE_COV.values).max(),
        np.linalg.eigvalsh(SINGLE_INDEX_COV.values).max()
    ]
})

Table_P2b_Single_Index_Parameters = pd.DataFrame({
    "Ticker": CANDIDATE_TICKERS,
    "Beta": [si_betas[t] for t in CANDIDATE_TICKERS],
    "ResidualVarianceMonthly": [
        si_residual_variances[t] for t in CANDIDATE_TICKERS
    ]
})

display(Table_P2_Risk_Model_Diagnostics)
display(Table_P2b_Single_Index_Parameters)


In [ ]:
# ============================================================
# 04D. PORTFOLIO FUNCTIONS AND BASE CONSTRAINTS
# ============================================================

BASE_BOUNDS = [
    (0.0, MAX_ETF_WEIGHT if ticker in ETFS else MAX_STOCK_WEIGHT)
    for ticker in CANDIDATE_TICKERS
]

def portfolio_statistics(weights, expected_returns, covariance):
    mu = np.asarray(expected_returns)
    sigma = np.asarray(covariance)

    expected_return = weights @ mu
    variance = weights @ sigma @ weights
    volatility = np.sqrt(variance)

    return expected_return, volatility, variance

def optimize_mean_variance_utility(
    expected_returns,
    covariance,
    risk_aversion,
    bounds
):
    mu = np.asarray(expected_returns)
    sigma = np.asarray(covariance)
    n_assets = len(mu)

    def negative_utility(weights):
        expected_return = weights @ mu
        variance = weights @ sigma @ weights
        return -(expected_return - 0.5 * risk_aversion * variance)

    result = minimize(
        negative_utility,
        np.repeat(1 / n_assets, n_assets),
        method="SLSQP",
        bounds=bounds,
        constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
        options={"maxiter": 1000, "ftol": 1e-12}
    )

    if not result.success:
        raise RuntimeError(f"Optimization failed: {result.message}")

    return result.x

def optimize_max_sharpe(
    expected_returns,
    covariance,
    risk_free_rate,
    bounds
):
    mu = np.asarray(expected_returns)
    sigma = np.asarray(covariance)
    n_assets = len(mu)

    def negative_sharpe(weights):
        port_return = weights @ mu
        port_vol = np.sqrt(weights @ sigma @ weights)
        if port_vol <= 0:
            return 1e6
        return -((port_return - risk_free_rate) / port_vol)

    result = minimize(
        negative_sharpe,
        np.repeat(1 / n_assets, n_assets),
        method="SLSQP",
        bounds=bounds,
        constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
        options={"maxiter": 1000, "ftol": 1e-12}
    )

    if not result.success:
        raise RuntimeError(f"Max-Sharpe optimization failed: {result.message}")

    return result.x

Table_P3_Constraint_Policy = pd.DataFrame({
    "Constraint": [
        "Long only",
        "Fully invested growth sleeve",
        "Maximum stock weight",
        "Maximum ETF weight",
        "Borrowing",
        "Short selling",
        "Derivatives"
    ],
    "BasePolicy": [
        "Yes", "Yes",
        MAX_STOCK_WEIGHT, MAX_ETF_WEIGHT,
        "Prohibited", "Prohibited", "Prohibited"
    ],
    "Reason": [
        "Mandate",
        "Compare complete growth allocations on same capital base",
        "Limit single-name estimation/concentration risk",
        "Allow diversified ETF core without dominating entire sleeve",
        "Mandate",
        "Mandate",
        "Mandate"
    ]
})

display(Table_P3_Constraint_Policy)


In [ ]:
# ============================================================
# 04E. SAME DECISION UNDER TWO RISK MODELS
# ============================================================

mu_base = BASE_EXPECTED_RETURNS.loc[CANDIDATE_TICKERS].values

sample_weights = optimize_mean_variance_utility(
    mu_base, SAMPLE_COV.values, BASE_RISK_AVERSION, BASE_BOUNDS
)

single_index_weights = optimize_mean_variance_utility(
    mu_base, SINGLE_INDEX_COV.values, BASE_RISK_AVERSION, BASE_BOUNDS
)

Table_P4_Risk_Model_Allocation = pd.DataFrame({
    "Ticker": CANDIDATE_TICKERS,
    "SampleCovWeight": sample_weights,
    "SingleIndexWeight": single_index_weights
})

Table_P4_Risk_Model_Allocation["WeightDifference"] = (
    Table_P4_Risk_Model_Allocation["SingleIndexWeight"]
    - Table_P4_Risk_Model_Allocation["SampleCovWeight"]
)

risk_model_turnover = (
    0.5 * np.abs(sample_weights - single_index_weights).sum()
)

Table_P4b_Risk_Model_Robustness = pd.DataFrame({
    "Metric": [
        "Allocation turnover between risk models",
        "Sample covariance HHI",
        "Single-index HHI"
    ],
    "Value": [
        risk_model_turnover,
        np.sum(sample_weights ** 2),
        np.sum(single_index_weights ** 2)
    ]
})

display(Table_P4_Risk_Model_Allocation)
display(Table_P4b_Risk_Model_Robustness)


In [ ]:
# ============================================================
# 04F. RISK-AVERSION, EXPECTED-RETURN AND CONSTRAINT SENSITIVITY
# ============================================================

# Risk-aversion sensitivity under both required risk models
risk_aversion_rows = []
for model_name, covariance in {
    "SampleCovariance": SAMPLE_COV.values,
    "SingleIndex": SINGLE_INDEX_COV.values
}.items():
    for A in RISK_AVERSION_VALUES:
        w = optimize_mean_variance_utility(
            mu_base, covariance, A, BASE_BOUNDS
        )
        er, vol, var = portfolio_statistics(w, mu_base, covariance)
        risk_aversion_rows.append({
            "RiskModel": model_name,
            "RiskAversion": A,
            "ExpectedReturn": er,
            "Volatility": vol,
            "Utility": er - 0.5 * A * var,
            "LargestWeight": w.max()
        })

Table_P5_Risk_Aversion_Sensitivity = pd.DataFrame(risk_aversion_rows)

# Expected-return sensitivity: change only mu
mu_historical = ALTERNATIVE_EXPECTED_RETURNS.loc[
    CANDIDATE_TICKERS
].values

historical_forecast_weights = optimize_mean_variance_utility(
    mu_historical,
    SINGLE_INDEX_COV.values,
    BASE_RISK_AVERSION,
    BASE_BOUNDS
)

expected_return_turnover = (
    0.5 * np.abs(
        historical_forecast_weights - single_index_weights
    ).sum()
)

Table_P5b_Expected_Return_Sensitivity = pd.DataFrame({
    "Ticker": CANDIDATE_TICKERS,
    "CAPMStyleWeight": single_index_weights,
    "HistoricalMeanWeight": historical_forecast_weights,
    "WeightDifference": (
        historical_forecast_weights - single_index_weights
    )
})

# Concentration-policy sensitivity
constraint_cases = {
    "Tighter": {
        t: (0.0, 0.25 if t in ETFS else 0.10)
        for t in CANDIDATE_TICKERS
    },
    "Base": {
        t: (0.0, MAX_ETF_WEIGHT if t in ETFS else MAX_STOCK_WEIGHT)
        for t in CANDIDATE_TICKERS
    },
    "Looser": {
        t: (0.0, 0.40 if t in ETFS else 0.20)
        for t in CANDIDATE_TICKERS
    }
}

constraint_rows = []
for name, bound_map in constraint_cases.items():
    bounds = [bound_map[t] for t in CANDIDATE_TICKERS]
    w = optimize_mean_variance_utility(
        mu_base,
        SINGLE_INDEX_COV.values,
        BASE_RISK_AVERSION,
        bounds
    )
    er, vol, _ = portfolio_statistics(
        w, mu_base, SINGLE_INDEX_COV.values
    )
    constraint_rows.append({
        "ConstraintCase": name,
        "ExpectedReturn": er,
        "Volatility": vol,
        "LargestWeight": w.max(),
        "HHI": np.sum(w ** 2)
    })

Table_P5c_Constraint_Sensitivity = pd.DataFrame(constraint_rows)

display(Table_P5_Risk_Aversion_Sensitivity)
print(f"Turnover from expected-return specification: {expected_return_turnover:.2%}")
display(Table_P5b_Expected_Return_Sensitivity)
display(Table_P5c_Constraint_Sensitivity)


In [ ]:
# ============================================================
# 04G. ETA ALTERNATIVE AND COMMON-EVALUATION COMPARISON
# ============================================================

historical_mu = candidate_returns.mean().values * 12
historical_rf = rf_monthly.mean() * 12

eta_weights = optimize_max_sharpe(
    historical_mu,
    SAMPLE_COV.values,
    historical_rf,
    BASE_BOUNDS
)

decision_candidates = {
    "Utility_SampleCov": sample_weights,
    "Utility_SingleIndex": single_index_weights,
    "Eta_Historical_MaxSharpe": eta_weights
}

candidate_rows = []

# Evaluate every candidate under the same base mu and single-index covariance
for strategy, weights in decision_candidates.items():
    er, vol, var = portfolio_statistics(
        weights, mu_base, SINGLE_INDEX_COV.values
    )
    candidate_rows.append({
        "Strategy": strategy,
        "BaseExpectedReturn": er,
        "BaseVolatility": vol,
        "BaseSharpe": (er - BASE_RF_ANNUAL) / vol,
        "BaseUtility": (
            er - 0.5 * BASE_RISK_AVERSION * var
        ),
        "LargestWeight": weights.max(),
        "HHI": np.sum(weights ** 2)
    })

Table_P6_Decision_Candidates = pd.DataFrame(candidate_rows)

Table_P6b_Strategy_Weights = pd.DataFrame({
    "Ticker": CANDIDATE_TICKERS,
    "Utility_SampleCov": sample_weights,
    "Utility_SingleIndex": single_index_weights,
    "Eta_Historical_MaxSharpe": eta_weights
})

PROVISIONAL_GROWTH_STRATEGY = "Utility_SingleIndex"
CREDIBLE_REJECTED_ALTERNATIVE = "Eta_Historical_MaxSharpe"

provisional_weights = decision_candidates[PROVISIONAL_GROWTH_STRATEGY]

display(Table_P6_Decision_Candidates)
display(Table_P6b_Strategy_Weights)


In [ ]:
# ============================================================
# 04H. FULL USD 12M INITIAL ALLOCATION, INCLUDING TRADING COST
# ============================================================

# Residual capital must fund both growth holdings and initial trading costs.
# If cost = c times purchased notional:
#   purchased notional + c*purchased notional = residual budget.

GROWTH_INVESTED_NOTIONAL = (
    GROWTH_BUDGET_BEFORE_COSTS / (1 + BASE_TRADING_COST)
)

INITIAL_TRADING_COST = (
    GROWTH_INVESTED_NOTIONAL * BASE_TRADING_COST
)

growth_dollar_allocations = (
    provisional_weights * GROWTH_INVESTED_NOTIONAL
)

allocation_rows = [
    {
        "Sleeve": "2024 Liability",
        "SecurityOrUse": "Beta",
        "DollarAllocation": beta_initial_cost,
        "Role": "Fund 2024 scholarship payment"
    },
    {
        "Sleeve": "2027 Liability",
        "SecurityOrUse": "Delta",
        "DollarAllocation": delta_initial_cost,
        "Role": "Fund 2027 learning-centre payment"
    }
]

for ticker, amount in zip(
    CANDIDATE_TICKERS, growth_dollar_allocations
):
    allocation_rows.append({
        "Sleeve": "Growth",
        "SecurityOrUse": ticker,
        "DollarAllocation": amount,
        "Role": "2030 real-value objective"
    })

allocation_rows.append({
    "Sleeve": "Implementation",
    "SecurityOrUse": "Initial trading cost",
    "DollarAllocation": INITIAL_TRADING_COST,
    "Role": "10 bps one-way growth implementation cost"
})

Table_P7_Full_Allocation = pd.DataFrame(allocation_rows)
Table_P7_Full_Allocation["PercentOfTotalFund"] = (
    Table_P7_Full_Allocation["DollarAllocation"] / INITIAL_CAPITAL
)

allocation_total = Table_P7_Full_Allocation["DollarAllocation"].sum()

if not np.isclose(allocation_total, INITIAL_CAPITAL, atol=0.01):
    raise ValueError("Full initial allocation does not reconcile to USD 12m.")

display(Table_P7_Full_Allocation)
print(f"Full initial use of capital: USD {allocation_total:,.2f}")


In [ ]:
# ============================================================
# 04I. TRUSTEE 15% LOSS CONCERN AND IMPLEMENTATION SENSITIVITY
# ============================================================

provisional_row = Table_P6_Decision_Candidates.loc[
    Table_P6_Decision_Candidates["Strategy"]
    == PROVISIONAL_GROWTH_STRATEGY
].iloc[0]

z_5pct = norm.ppf(0.05)

growth_5pct_return = (
    provisional_row["BaseExpectedReturn"]
    + z_5pct * provisional_row["BaseVolatility"]
)

growth_share_of_total = GROWTH_INVESTED_NOTIONAL / INITIAL_CAPITAL

# Approximate total-fund downside contribution from the growth sleeve only.
# Bond mark-to-market risk is analysed separately in Section 02.
approx_total_fund_5pct_impact = (
    growth_share_of_total * growth_5pct_return
)

Table_P8_Risk_Tolerance = pd.DataFrame({
    "Metric": [
        "Trustee reference loss level",
        "Growth share of initial fund",
        "Approximate 5% one-year growth return",
        "Approximate growth-driven total-fund impact"
    ],
    "Value": [
        -0.15,
        growth_share_of_total,
        growth_5pct_return,
        approx_total_fund_5pct_impact
    ],
    "Interpretation": [
        "Reference concern only; not a formal hard limit",
        "Liability reserve is separated first",
        "Normal approximation; diagnostic, not guarantee",
        "Excludes one-year bond MTM movement; Section 02 covers rate shocks"
    ]
})

trading_cost_rows = []
for cost_rate in [0.0000, 0.0010, 0.0025]:
    invested = GROWTH_BUDGET_BEFORE_COSTS / (1 + cost_rate)
    cost = invested * cost_rate
    trading_cost_rows.append({
        "TradingCostRate": cost_rate,
        "GrowthInvestedNotional": invested,
        "InitialDollarCost": cost,
        "CostPctOfTotalFund": cost / INITIAL_CAPITAL
    })

Table_P8b_Trading_Cost_Sensitivity = pd.DataFrame(trading_cost_rows)

display(Table_P8_Risk_Tolerance)
display(Table_P8b_Trading_Cost_Sensitivity)


In [ ]:
# ============================================================
# 04J. IMPLEMENTATION POLICY, DECISION SUMMARY AND EXPORT
# ============================================================

Table_P9_Implementation_Policy = pd.DataFrame({
    "PolicyItem": [
        "Liability priority",
        "Growth strategy at 31 Dec 2022",
        "Rebalancing frequency",
        "Rebalancing timing",
        "Maximum stock weight",
        "Maximum ETF weight",
        "Borrowing",
        "Short selling",
        "Derivatives",
        "Base trading cost",
        "Status of recommendation"
    ],
    "Policy": [
        "Fund mandatory payments before growth allocation",
        PROVISIONAL_GROWTH_STRATEGY,
        "Annual",
        "31 December, after any scheduled liability payment",
        f"{MAX_STOCK_WEIGHT:.0%}",
        f"{MAX_ETF_WEIGHT:.0%}",
        "Prohibited",
        "Prohibited",
        "Prohibited",
        f"{BASE_TRADING_COST:.2%} one-way",
        "Provisional until factor and 2023-2025 frozen evaluation are completed"
    ]
})

Section_04_Summary = pd.DataFrame({
    "Item": [
        "Candidate assets",
        "Base expected-return framework",
        "Base risk model",
        "Base risk aversion",
        "Provisional growth strategy",
        "Credible rejected alternative",
        "Risk-model allocation turnover",
        "Expected-return-model turnover",
        "Initial liability reserve",
        "Growth invested notional",
        "Initial trading cost",
        "Full initial use of capital"
    ],
    "Value": [
        len(CANDIDATE_TICKERS),
        "CAPM-style common systematic forecast",
        "Single-index",
        BASE_RISK_AVERSION,
        PROVISIONAL_GROWTH_STRATEGY,
        CREDIBLE_REJECTED_ALTERNATIVE,
        risk_model_turnover,
        expected_return_turnover,
        LIABILITY_BUCKET_COST,
        GROWTH_INVESTED_NOTIONAL,
        INITIAL_TRADING_COST,
        allocation_total
    ]
})

section_04_outputs = {
    "Table_P1_Expected_Return_Inputs": Table_P1_Expected_Return_Inputs.reset_index(),
    "Table_P1b_Return_Assumptions": Table_P1b_Return_Assumptions,
    "Table_P2_Risk_Model_Diagnostics": Table_P2_Risk_Model_Diagnostics,
    "Table_P2b_Single_Index_Parameters": Table_P2b_Single_Index_Parameters,
    "Table_P3_Constraint_Policy": Table_P3_Constraint_Policy,
    "Table_P4_Risk_Model_Allocation": Table_P4_Risk_Model_Allocation,
    "Table_P4b_Risk_Model_Robustness": Table_P4b_Risk_Model_Robustness,
    "Table_P5_Risk_Aversion_Sensitivity": Table_P5_Risk_Aversion_Sensitivity,
    "Table_P5b_Expected_Return_Sensitivity": Table_P5b_Expected_Return_Sensitivity,
    "Table_P5c_Constraint_Sensitivity": Table_P5c_Constraint_Sensitivity,
    "Table_P6_Decision_Candidates": Table_P6_Decision_Candidates,
    "Table_P6b_Strategy_Weights": Table_P6b_Strategy_Weights,
    "Table_P7_Full_Allocation": Table_P7_Full_Allocation,
    "Table_P8_Risk_Tolerance": Table_P8_Risk_Tolerance,
    "Table_P8b_Trading_Cost_Sensitivity": Table_P8b_Trading_Cost_Sensitivity,
    "Table_P9_Implementation_Policy": Table_P9_Implementation_Policy,
    "Section_04_Summary": Section_04_Summary
}

for name, table in section_04_outputs.items():
    table.to_csv(TABLE_DIR / f"{name}.csv", index=False)

SAMPLE_COV.to_csv(TABLE_DIR / "Matrix_P_Sample_Covariance.csv")
SINGLE_INDEX_COV.to_csv(TABLE_DIR / "Matrix_P_Single_Index_Covariance.csv")

display(Table_P9_Implementation_Policy)
display(Section_04_Summary)
print("Sections 01-04 upgraded and exported.")
